# bdap_lea - notebook v0

Validazione rapida + analisi esplorativa della pipeline **raw → clean → mart**.

- i check tecnici (righe, null, readiness) sono delegati al toolkit
- le SQL sono la fonte di verità: leggile prima di interpretare i numeri
- il notebook serve l'analisi, non sostituisce la validazione della pipeline
- evitare output pesanti o immagini embeddate nel commit

In [ ]:
import json, subprocess, sys, duckdb, yaml
from pathlib import Path

# --- Unico parametro da impostare ---
SLUG = "bdap_lea"

# --- Trova dataset.yml e parsa anno ---
_start = Path.cwd().resolve()
for probe in [_start, *_start.parents]:
    if (probe / "dataset.yml").exists():
        CANDIDATE_DIR = probe
        break
else:
    raise FileNotFoundError(f"dataset.yml non trovato risalendo da {_start}")

CFG_PATH = CANDIDATE_DIR / "dataset.yml"
SQL_DIR  = CANDIDATE_DIR / "sql"
_cfg_raw = yaml.safe_load(CFG_PATH.read_text())
ANNO = _cfg_raw["dataset"]["years"][-1]

def tk(*args: str) -> dict:
    """Esegue toolkit e torna JSON."""
    cmd = ["toolkit", *args, "-c", str(CFG_PATH), "--json"]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr, file=sys.stderr)
        r.check_returncode()
    return json.loads(r.stdout) if r.stdout.strip() else {}

def tk_year(*args: str) -> dict:
    """Chiamata con --year per comandi che lo richiedono."""
    return tk(*args, "--year", str(ANNO))

def show(df) -> None:
    """Stampa un DataFrame: display() in Jupyter, print() altrove."""
    try:
        display(df)
    except NameError:
        print(df.to_string())

WORKSPACE_ROOT = CANDIDATE_DIR.parent.parent  # dataset-incubator/
def _rel(p: Path) -> str:
    """Path relativo al workspace, per output leggibile e portabile."""
    try: return str(p.relative_to(WORKSPACE_ROOT))
    except ValueError: return str(p)

print(f"slug: {SLUG}    anno: {ANNO}")
print(f"config: {_rel(CFG_PATH)}")

# Paths dei layer (con anno esplicito)
paths = tk_year("inspect", "paths")
P = paths.get("paths", {})
RAW_DIR   = Path(P.get("raw", {}).get("dir", ""))
CLEAN_DIR = Path(P.get("clean", {}).get("dir", ""))
MART_DIR  = Path(P.get("mart", {}).get("dir", ""))
MART_OUTPUTS = P.get("mart", {}).get("outputs", [])
MART_PATH = Path(MART_OUTPUTS[0]) if MART_OUTPUTS else None
MART_TABLE = MART_PATH.stem if MART_PATH else None
print(f"raw:   {_rel(RAW_DIR)}")
print(f"clean: {_rel(CLEAN_DIR)}")
print(f"mart:  {_rel(MART_DIR)}")

## SQL di riferimento

Le SQL sono la fonte di verità per capire cosa deve contenere ogni layer.
Leggile prima di interpretare i check numerici.

In [ ]:
for sql_file in sorted(SQL_DIR.glob("*.sql")):
    print(f"{'='*60}")
    print(f"  {sql_file.name}")
    print(f"{'='*60}")
    print(sql_file.read_text())
    print()

## Quality gates

Il toolkit ha già eseguito run + validate + readiness. Qui si riassume il risultato.

In [ ]:
run_info = paths.get("latest_run", {})
print(f"Ultimo run: {run_info.get('status','?')}  (ID: {run_info.get('run_id','?')})")
print()

# Schema clean
schema_clean = tk("inspect", "schema", "-l", "clean")
cols_clean = schema_clean.get("schema", [])
print(f"Clean: {schema_clean.get('columns', 0)} colonne")
for c in cols_clean[:5]:
    print(f"  {c.get('name','?'):35s} {c.get('type','?')}")
if len(cols_clean) > 5: print(f"  ... ({len(cols_clean)} totali)")
print()

# Schema mart
schema_mart = tk("inspect", "schema", "-l", "mart")
cols_mart = schema_mart.get("schema", [])
print(f"Mart: {schema_mart.get('columns', 0)} colonne")
for c in cols_mart[:5]:
    print(f"  {c.get('name','?'):35s} {c.get('type','?')}")
if len(cols_mart) > 5: print(f"  ... ({len(cols_mart)} totali)")
print()

# Profili layer da inspect paths
profiles = paths.get("layer_profiles", {})
clean_prof = profiles.get("clean_output", {})
if clean_prof:
    print(f"Clean: {clean_prof.get('row_count','?')} righe, {clean_prof.get('columns_count','?')} colonne")
for tbl in profiles.get("mart_tables", []):
    print(f"Mart: {tbl.get('row_count','?')} righe ({tbl.get('name','?')})")
for trans in profiles.get("clean_to_mart", []):
    print(f"Transizione: {trans.get('source_row_count','?')} -> {trans.get('target_row_count','?')} righe")
    print(f"  rimosse: {trans.get('removed_columns', [])}, aggiunte: {trans.get('added_columns', [])}")
print()

# Raw hints (encoding, delim, etc.)
rh = paths.get("raw_hints", {})
print(f"Raw: {rh.get('primary_output_file','?')}  |  encoding={rh.get('encoding','?')}  delim={rh.get('delim','?')}  skip={rh.get('skip','?')}")
for w in rh.get('warnings', []):
    print(f"  warning: {w}")
print()

# Review readiness
readiness = tk("review-readiness")
rd = readiness.get("readiness", "?")
icon = {"ready": "✅", "needs-review": "⚠️", "incomplete": "🔴"}.get(rd, "?")
print(f"Readiness: {icon} {rd}")
print(f"Checks: {readiness.get('ok_count',0)}/{readiness.get('check_count',0)} ok, {readiness.get('fail_count',0)} falliti")

## Anteprima dati

Un colpo d'occhio sui dati del mart. Query esplorative, non analisi.
Personalizza in base al candidate.

In [ ]:
con = duckdb.connect()
if MART_PATH and MART_PATH.exists():
    p = str(MART_PATH).replace("'", "''")
    con.execute(f"CREATE OR REPLACE VIEW mart AS SELECT * FROM read_parquet('{p}')")
    cnt = con.execute("SELECT count(*) FROM mart").fetchone()[0]
    print(f"Mart: {MART_PATH.name}  |  {cnt} righe")
else:
    print("Mart non disponibile. Esegui: toolkit run mart")

In [ ]:
# --- Query di esempio ---
# Sostituisci con la query specifica del candidate
if MART_PATH:
    df = con.execute("SELECT * FROM mart LIMIT 10").fetchdf()
    show(df)
else:
    print("Esegui il mart prima di fare analisi.")

In [ ]:
# Profilo null rapido sul mart (prime 8 colonne)
if MART_PATH:
    cols = con.execute("DESCRIBE mart").fetchall()
    exprs = []
    for c in cols[:8]:
        safe = c[0].replace('"', '""')
        exprs.append(f'round(100.0 * sum(CASE WHEN "{safe}" IS NULL THEN 1 ELSE 0 END) / count(*), 2) as "null_{c[0]}"')
    if exprs:
        q = 'SELECT count(*) as tot, ' + ', '.join(exprs) + ' FROM mart'
        nulls = con.execute(q).fetchdf()
        show(nulls)

## Boundary e note

Documenta scelte di perimetro, esclusioni, anomalie.

In [ ]:
PERIMETRO = "..."  # descrizione del perimetro

print(f"Slug      : {SLUG}")
print(f"Perimetro : {PERIMETRO}")
print()
print("Note boundary:")
print("  -")

## Verdetto

Riepilogo end-to-end della pipeline, come `run-candidate`.

In [ ]:
run_info = paths.get("latest_run", {})
run_status = run_info.get("status", "?")
rd = readiness.get("readiness", "?")
profiles = paths.get("layer_profiles", {})
clean_p = profiles.get("clean_output", {})
mart_tables = profiles.get("mart_tables", [])
transitions = profiles.get("clean_to_mart", [])

pipeline_icon = {'SUCCESS': '✅', 'FAILED': '❌', 'IN_PROGRESS': '⏳'}.get(run_status, '?')
rd_icon = {"ready": "✅", "needs-review": "⚠️", "incomplete": "🔴"}.get(rd, "?")
print(f"Pipeline  : {pipeline_icon} {run_status}")
print(f"Readiness : {rd_icon} {rd}")
print()
if clean_p:
    print(f"  Clean: {clean_p.get('row_count', '?'):>8} righe, {clean_p.get('columns_count', '?')} colonne")
for tbl in mart_tables:
    print(f"  Mart:  {tbl.get('row_count', '?'):>8} righe ({tbl.get('name', '?')})")
for t in transitions:
    print(f"  Transizione: {t.get('source_row_count','?')} → {t.get('target_row_count','?')} righe")
    if t.get("removed_columns"):
        print(f"    colonne rimosse: {t['removed_columns']}")
    if t.get("added_columns"):
        print(f"    colonne aggiunte: {t['added_columns']}")
print()
if rd == "ready":
    print("✅ Notebook v0 completato — pipeline verificata, dati coerenti.")
elif rd == "needs-review":
    print("⚠️  Qualche anomalia — verificare i check falliti prima del merge.")
else:
    print("🔴 Candidate non pronto — risolvere i check falliti.")